# Xia et al. (2025) CVaR 配电网风险调度：方法级复现

论文：**Source-network-load-storage collaborated two-stage power dispatch of active distribution network with conditional value-at-risk**  
期刊：*International Journal of Electrical Power & Energy Systems*, 172 (2025) 111120  
DOI: `10.1016/j.ijepes.2025.111120`

本 Notebook 是**完整、自包含的复现入口**，不再通过 `%run` 隐藏核心代码。它复现论文最关键的实验链路：

> **风/光/负荷预测误差 → 1000 个不确定场景 → 20 个典型场景 → IEEE 33 节点 → 储能功率序列 → CVaR 尾部风险 → 风险厌恶调度**

目标是 method-level clean-room reproduction：跑通论文的核心机制，并明确论文公开参数与复现假设的边界。


## 0. 数据来源与复现边界

这篇论文没有公开原始风/光/负荷数据，也没有找到作者公开的官方代码仓库；论文 Data availability 写的是 **“Data will be made available on request”**。

因此本复现**没有使用某个现成真实时序数据集**。数据由两部分组成：

1. **网络数据**：标准 IEEE 33-bus / MATPOWER `case33bw` 拓扑、节点负荷与线路参数；
2. **时序与不确定性数据**：依据论文 Fig. 8 的负荷、PV、WT 日变化趋势构造 24 h 基准曲线，再按论文“预测误差 → 场景生成 → 场景削减”的思路合成 1000 个相关场景，并缩减到 20 个典型场景。

保留的论文设置包括：WT@17/32（各 0.9 MW）、PV@21/24（各 0.6 MW）、ESS@15（1.8 MWh / 0.3 MW）、ESS@32（1.0 MWh / 0.2 MW）、24 h/1 h、论文峰谷电价，以及 Case 2/Case 3 的“平均目标 vs. CVaR 风险目标”比较逻辑。

主要替代项：Gaussian copula 代替论文 PED 相关性处理；LinDistFlow 代替完整 DistFlow+MISOCP；不复现 OLTC/CB/SOP/SVC/DR/网络重构；论文未公开数值 `β`，这里默认 `β=0.95`。


## 1. IEEE 33 节点、24 h 基准曲线与 1000→20 场景

下面这一个代码单元包含**完整的数据构造与场景生成代码**：标准 IEEE 33-bus、论文设备位置、24 h 基准曲线、LHS + Gaussian copula 相关场景、fast-forward reduction，以及 LinDistFlow 电压灵敏度。


In [ ]:
"""Scenario and network setup for Xia et al. (2025) method-level reproduction."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import qmc, norm, t as student_t
from scipy.spatial.distance import cdist

np.set_printoptions(precision=4, suppress=True)
SEED = 123
rng = np.random.default_rng(SEED)

# Standard IEEE 33-bus data (MW / MVAr; branch R/X in ohm)
loads = {
2:(0.100,0.060),3:(0.090,0.040),4:(0.120,0.080),5:(0.060,0.030),6:(0.060,0.020),
7:(0.200,0.100),8:(0.200,0.100),9:(0.060,0.020),10:(0.060,0.020),11:(0.045,0.030),
12:(0.060,0.035),13:(0.060,0.035),14:(0.120,0.080),15:(0.060,0.010),16:(0.060,0.020),
17:(0.060,0.020),18:(0.090,0.040),19:(0.090,0.040),20:(0.090,0.040),21:(0.090,0.040),
22:(0.090,0.040),23:(0.090,0.050),24:(0.420,0.200),25:(0.420,0.200),26:(0.060,0.025),
27:(0.060,0.025),28:(0.060,0.020),29:(0.120,0.070),30:(0.200,0.600),31:(0.150,0.070),
32:(0.210,0.100),33:(0.060,0.040)
}
branches = [
(1,2,0.0922,0.0470),(2,3,0.4930,0.2511),(3,4,0.3660,0.1864),(4,5,0.3811,0.1941),
(5,6,0.8190,0.7070),(6,7,0.1872,0.6188),(7,8,1.7114,1.2351),(8,9,1.0300,0.7400),
(9,10,1.0440,0.7400),(10,11,0.1966,0.0650),(11,12,0.3744,0.1238),(12,13,1.4680,1.1550),
(13,14,0.5416,0.7129),(14,15,0.5910,0.5260),(15,16,0.7463,0.5450),(16,17,1.2890,1.7210),
(17,18,0.7320,0.5740),(2,19,0.1640,0.1565),(19,20,1.5042,1.3554),(20,21,0.4095,0.4784),
(21,22,0.7089,0.9373),(3,23,0.4512,0.3083),(23,24,0.8980,0.7091),(24,25,0.8960,0.7011),
(6,26,0.2030,0.1034),(26,27,0.2842,0.1447),(27,28,1.0590,0.9337),(28,29,0.8042,0.7006),
(29,30,0.5075,0.2585),(30,31,0.9744,0.9630),(31,32,0.3105,0.3619),(32,33,0.3410,0.5302),
]

children={i:[] for i in range(1,34)}
parent={}
for a,b,r,x in branches:
    children[a].append(b)
    parent[b]=a

def descendants(i):
    out=[i]
    for c in children[i]:
        out.extend(descendants(c))
    return out

desc_sets={i:set(descendants(i)) for i in range(1,34)}
baseKV=12.66
baseMVA=10.0
zbase=baseKV**2/baseMVA
P0=np.zeros(34)
Q0=np.zeros(34)
for i,(p,q) in loads.items():
    P0[i]=p
    Q0[i]=q

print('Total P/Q load =', P0.sum(), 'MW /', Q0.sum(), 'MVAr')
print('Paper DG/ESS placements: WT@17,32; PV@21,24; ESS@15,32')

# Fig. 8-inspired 24 h profiles. The paper does not publish the raw series.
hours=np.arange(1,25)
load_scale=np.array([0.62,0.58,0.56,0.55,0.58,0.66,0.76,0.86,0.94,1.00,1.02,0.99,
                     0.91,0.86,0.84,0.88,0.94,1.02,1.08,1.05,0.96,0.88,0.78,0.69])
pv_cf=np.array([0,0,0,0,0,0.02,0.12,0.30,0.52,0.72,0.88,0.96,0.92,0.78,0.58,0.36,0.15,0.03,0,0,0,0,0,0])
wind_cf=np.array([0.62,0.58,0.55,0.52,0.50,0.48,0.52,0.58,0.64,0.68,0.72,0.70,
                  0.66,0.62,0.60,0.58,0.61,0.66,0.72,0.78,0.82,0.78,0.72,0.66])
price=np.array([400]*8+[800,800,1200,1200,800,800,800,1200,1200,1200,1200,1200,1200,1200,800,800],dtype=float)

fig,ax=plt.subplots(figsize=(10,4))
ax.plot(hours, load_scale, marker='o', label='Load factor')
ax.plot(hours, pv_cf, marker='o', label='PV factor')
ax.plot(hours, wind_cf, marker='o', label='WT factor')
ax.set_xlabel('Hour')
ax.set_ylabel('p.u. factor')
ax.grid(alpha=.25)
ax.legend()
plt.show()

# 1000 spatiotemporally correlated scenarios via LHS + Gaussian copula.
pv_caps=np.array([0.6,0.6])
wind_caps=np.array([0.9,0.9])
n_raw=1000
D=5*24
sampler=qmc.LatinHypercube(d=D, seed=SEED)
U=sampler.random(n_raw)
Z=norm.ppf(U)

rho_t=.65
Tcorr=rho_t**np.abs(np.subtract.outer(np.arange(24),np.arange(24)))
Vcorr=np.array([
[1.0,.75,.20,.15,.10],
[.75,1.0,.15,.20,.10],
[.20,.15,1.0,.65,.10],
[.15,.20,.65,1.0,.10],
[.10,.10,.10,.10,1.0],
])
C=np.kron(Tcorr,Vcorr)
L=np.linalg.cholesky(C+1e-8*np.eye(D))
Zc=Z@L.T
Uc=np.clip(norm.cdf(Zc),1e-6,1-1e-6).reshape(n_raw,24,5)

pv_sigma=0.08+0.08*pv_cf
wind_sigma=np.full(24,0.14)
load_sigma=0.04+0.01*(load_scale>0.95)
df=5
pv_scen=np.zeros((n_raw,24,2))
wind_scen=np.zeros((n_raw,24,2))
load_scen=np.zeros((n_raw,24))
for h in range(24):
    for k in range(2):
        e=student_t.ppf(Uc[:,h,k],df=df)*pv_sigma[h]*pv_caps[k]
        pv_scen[:,h,k]=np.clip(pv_cf[h]*pv_caps[k]+e,0,pv_caps[k])
    for k in range(2):
        e=student_t.ppf(Uc[:,h,2+k],df=df)*wind_sigma[h]*wind_caps[k]
        wind_scen[:,h,k]=np.clip(wind_cf[h]*wind_caps[k]+e,0,wind_caps[k])
    e=norm.ppf(Uc[:,h,4])*load_sigma[h]
    load_scen[:,h]=np.clip(load_scale[h]*(1+e),0.4,1.25)

print('raw scenarios:', pv_scen.shape[0])

X=np.concatenate([
    pv_scen.reshape(n_raw,-1)/0.6,
    wind_scen.reshape(n_raw,-1)/0.9,
    load_scen
],axis=1)

def fast_forward_reduce(X,k,probs=None):
    n=len(X)
    if probs is None:
        probs=np.full(n,1/n)
    D=cdist(X,X)
    selected=[]
    nearest=np.full(n,np.inf)
    candidates=set(range(n))
    for _ in range(k):
        best=None
        best_obj=np.inf
        best_nearest=None
        for j in candidates:
            candidate=np.minimum(nearest,D[:,j])
            obj=float(probs@candidate)
            if obj<best_obj:
                best_obj=obj
                best=j
                best_nearest=candidate
        selected.append(best)
        candidates.remove(best)
        nearest=best_nearest
    sel=np.array(selected)
    assignment=D[:,sel].argmin(axis=1)
    p_red=np.array([probs[assignment==j].sum() for j in range(k)])
    return sel,p_red,assignment

sel,p_red,assignment=fast_forward_reduce(X,20)
pvR=pv_scen[sel]
windR=wind_scen[sel]
loadR=load_scen[sel]
print('reduced scenarios:',len(sel),'probability sum:',p_red.sum(),'range:',(p_red.min(),p_red.max()))

net_load_plot=3.715*loadR-pvR.sum(axis=2)-windR.sum(axis=2)
fig,ax=plt.subplots(figsize=(10,4))
for s in range(20):
    ax.plot(hours,net_load_plot[s],alpha=.35)
ax.plot(hours,(p_red[:,None]*net_load_plot).sum(0),linewidth=2.5,label='Probability-weighted mean')
ax.set_xlabel('Hour')
ax.set_ylabel('System net load (MW)')
ax.grid(alpha=.25)
ax.legend()
plt.show()

# LinDistFlow approximation used instead of the paper's DistFlow + MISOCP.
def lindistflow_v2(Pnode,Qnode):
    v=np.ones(34)
    Pflow=np.zeros(len(branches))
    Qflow=np.zeros(len(branches))
    for ei,(a,b,r,x) in enumerate(branches):
        ds=desc_sets[b]
        P=Pnode[list(ds)].sum()
        Q=Qnode[list(ds)].sum()
        Pflow[ei]=P
        Qflow[ei]=Q
        v[b]=v[a]-2*((r/zbase)*(P/baseMVA)+(x/zbase)*(Q/baseMVA))
    return v,Pflow,Qflow

crit=[18,22,25,33]
ess_buses=[15,32]
sens=np.zeros((len(crit),2))
for k,bus in enumerate(ess_buses):
    p=np.zeros(34)
    q=np.zeros(34)
    p[bus]=1.0
    v,_,_=lindistflow_v2(p,q)
    sens[:,k]=v[crit]-1.0

S=20
T=24
base_v=np.zeros((S,T,len(crit)))
net_total=np.zeros((S,T))
for s in range(S):
    for h in range(T):
        P=P0*loadR[s,h]
        Q=Q0*loadR[s,h]
        P=P.copy()
        Q=Q.copy()
        P[21]-=pvR[s,h,0]
        P[24]-=pvR[s,h,1]
        P[17]-=windR[s,h,0]
        P[32]-=windR[s,h,1]
        v,_,_=lindistflow_v2(P,Q)
        base_v[s,h]=v[crit]
        net_total[s,h]=P.sum()

print('Without ESS, critical-bus voltage range:',np.sqrt(base_v).min(),np.sqrt(base_v).max())
print('Voltage sensitivity (squared-voltage change per +1 MW ESS charging load):')
print(pd.DataFrame(sens,index=[f'bus {b}' for b in crit],columns=['ESS@15','ESS@32']).to_string())


## 2. CVaR 如何进入储能序列搜索

优化变量本质上仍然是 24 h 的储能充放电序列。Case 2 只强调概率加权平均运行表现；Case 3 在同一搜索问题里加入尾部风险项。

直观上，候选序列 `x=[P_1,...,P_24]` 会在 20 个未来场景下产生一组经济损失和网络风险损失。CVaR 不负责生成场景，而是评价“坏尾部场景的平均损失”，然后把这个风险评价放进目标函数，使优化器愿意用少量平均经济性换取极端场景下更低的电压风险。

下面代码包含完整的 ESS SOC/功率约束、场景功率平衡、LinDistFlow 电压风险、CVaR 线性化、Case 1/2/3 比较和风险权重敏感性实验。


In [ ]:
"""CVaR ESS dispatch experiment for Xia et al. (2025) method-level reproduction."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog
from scipy.sparse import csr_matrix
from scenario_setup import p_red, price, base_v, sens, crit, net_total, hours

def solve_dispatch(lambda_risk=0.0,beta=0.95,kv=50000.0,cycle_cost=30.0,export_frac=0.15,vmin=0.94,vmax=1.06):
    S,T,K,B=20,24,2,len(crit)
    idx={}
    n=0
    def alloc(name,shape):
        nonlocal n
        arr=np.arange(n,n+int(np.prod(shape))).reshape(shape)
        idx[name]=arr
        n+=arr.size
        return arr

    pch=alloc('pch',(K,T))
    pdis=alloc('pdis',(K,T))
    soc=alloc('soc',(K,T+1))
    gimp=alloc('gimp',(S,T))
    gexp=alloc('gexp',(S,T))
    vlo=alloc('vlo',(S,T,B))
    vhi=alloc('vhi',(S,T,B))
    eta_c=alloc('eta_c',(1,))
    zc=alloc('zc',(S,))
    eta_v=alloc('eta_v',(1,))
    zv=alloc('zv',(S,))

    c=np.zeros(n)
    for s in range(S):
        ps=p_red[s]
        for h in range(T):
            c[gimp[s,h]]+=(1-lambda_risk)*ps*price[h]
            c[gexp[s,h]]+=(1-lambda_risk)*ps*(-export_frac*price[h])
            for k in range(K):
                c[pch[k,h]]+=(1-lambda_risk)*ps*cycle_cost
                c[pdis[k,h]]+=(1-lambda_risk)*ps*cycle_cost
            for b in range(B):
                c[vlo[s,h,b]]+=(1-lambda_risk)*ps*kv/B
                c[vhi[s,h,b]]+=(1-lambda_risk)*ps*kv/B

    # CVaR linear objective terms.
    c[eta_c[0]]+=lambda_risk
    c[zc]+=lambda_risk*p_red/(1-beta)
    c[eta_v[0]]+=lambda_risk
    c[zv]+=lambda_risk*p_red/(1-beta)

    # Equalities: SOC dynamics + per-scenario power balance.
    rr=[]
    cc=[]
    dd=[]
    beq=[]
    r=0
    caps=np.array([1.8,1.0])
    pmax=np.array([0.3,0.2])
    eta_ch=.95
    eta_dis=.95
    init=.5

    for k in range(K):
        rr.append(r);cc.append(soc[k,0]);dd.append(1);beq.append(caps[k]*init);r+=1
        for h in range(T):
            for col,val in [
                (soc[k,h+1],1),
                (soc[k,h],-1),
                (pch[k,h],-eta_ch),
                (pdis[k,h],1/eta_dis)
            ]:
                rr.append(r);cc.append(col);dd.append(val)
            beq.append(0);r+=1
        rr.append(r);cc.append(soc[k,T]);dd.append(1);beq.append(caps[k]*init);r+=1

    for s in range(S):
        for h in range(T):
            for col,val in [(gimp[s,h],1),(gexp[s,h],-1)]:
                rr.append(r);cc.append(col);dd.append(val)
            for k in range(K):
                rr += [r,r]
                cc += [pdis[k,h],pch[k,h]]
                dd += [1,-1]
            beq.append(net_total[s,h])
            r+=1

    Aeq=csr_matrix((dd,(rr,cc)),shape=(r,n))
    beq=np.array(beq)

    # Inequalities: voltage slacks + CVaR linearization.
    rr=[]
    cc=[]
    dd=[]
    bub=[]
    r=0
    lo2=vmin**2
    hi2=vmax**2

    for s in range(S):
        for h in range(T):
            for bi in range(B):
                # vlo >= lo2 - v2
                rr.append(r);cc.append(vlo[s,h,bi]);dd.append(-1)
                for k in range(K):
                    rr += [r,r]
                    cc += [pch[k,h],pdis[k,h]]
                    dd += [-sens[bi,k],sens[bi,k]]
                bub.append(base_v[s,h,bi]-lo2)
                r+=1

                # vhi >= v2 - hi2
                rr.append(r);cc.append(vhi[s,h,bi]);dd.append(-1)
                for k in range(K):
                    rr += [r,r]
                    cc += [pch[k,h],pdis[k,h]]
                    dd += [sens[bi,k],-sens[bi,k]]
                bub.append(hi2-base_v[s,h,bi])
                r+=1

    # Economic CVaR: z_s >= cost_s - eta.
    for s in range(S):
        for h in range(T):
            rr += [r,r]
            cc += [gimp[s,h],gexp[s,h]]
            dd += [price[h],-export_frac*price[h]]
            for k in range(K):
                rr += [r,r]
                cc += [pch[k,h],pdis[k,h]]
                dd += [cycle_cost,cycle_cost]
        rr += [r,r]
        cc += [eta_c[0],zc[s]]
        dd += [-1,-1]
        bub.append(0)
        r+=1

    # Network-risk CVaR.
    for s in range(S):
        for h in range(T):
            for bi in range(B):
                rr += [r,r]
                cc += [vlo[s,h,bi],vhi[s,h,bi]]
                dd += [kv/B,kv/B]
        rr += [r,r]
        cc += [eta_v[0],zv[s]]
        dd += [-1,-1]
        bub.append(0)
        r+=1

    Aub=csr_matrix((dd,(rr,cc)),shape=(r,n))
    bub=np.array(bub)

    bounds=[(0,None)]*n
    for k in range(K):
        for h in range(T):
            bounds[pch[k,h]]=(0,pmax[k])
            bounds[pdis[k,h]]=(0,pmax[k])
        for h in range(T+1):
            bounds[soc[k,h]]=(0.1*caps[k],caps[k])

    for s in range(S):
        for h in range(T):
            bounds[gimp[s,h]]=(0,10)
            bounds[gexp[s,h]]=(0,10)

    bounds[eta_c[0]]=(0,None)
    bounds[eta_v[0]]=(0,None)

    result=linprog(c,A_ub=Aub,b_ub=bub,A_eq=Aeq,b_eq=beq,bounds=bounds,method='highs')
    if not result.success:
        raise RuntimeError(result.message)

    pars=dict(
        kv=kv,
        cycle_cost=cycle_cost,
        export_frac=export_frac,
        beta=beta,
        lambda_risk=lambda_risk,
        vmin=vmin,
        vmax=vmax
    )
    return result,idx,pars

def weighted_cvar(vals,probs,beta):
    vals=np.asarray(vals)
    candidates=np.unique(vals)
    objectives=[
        eta+np.sum(probs*np.maximum(vals-eta,0))/(1-beta)
        for eta in candidates
    ]
    j=int(np.argmin(objectives))
    return float(objectives[j]),float(candidates[j])

def evaluate(result,idx,pars):
    x=result.x
    pch=x[idx['pch']]
    pdis=x[idx['pdis']]
    soc=x[idx['soc']]
    gimp=x[idx['gimp']]
    gexp=x[idx['gexp']]

    v2=np.zeros((20,24,len(crit)))
    for s in range(20):
        for h in range(24):
            v2[s,h]=base_v[s,h]+sens@(pch[:,h]-pdis[:,h])

    v=np.sqrt(np.maximum(v2,0))
    lo=np.maximum(0,pars['vmin']-v)
    hi=np.maximum(0,v-pars['vmax'])
    lo2=np.maximum(0,pars['vmin']**2-v2)
    hi2=np.maximum(0,v2-pars['vmax']**2)

    cost=np.array([
        np.sum(price*gimp[s]-pars['export_frac']*price*gexp[s])
        +pars['cycle_cost']*np.sum(pch+pdis)
        for s in range(20)
    ])
    risk=pars['kv']/len(crit)*np.sum(lo2+hi2,axis=(1,2))

    cc,_=weighted_cvar(cost,p_red,pars['beta'])
    cr,_=weighted_cvar(risk,p_red,pars['beta'])

    return dict(
        pch=pch,
        pdis=pdis,
        soc=soc,
        gimp=gimp,
        gexp=gexp,
        v=v,
        cost_s=cost,
        risk_s=risk,
        expected_cost=float(p_red@cost),
        expected_risk=float(p_red@risk),
        cvar_cost=cc,
        cvar_risk=cr,
        violation_pu_h=float(np.sum(p_red[:,None,None]*(lo+hi))),
        violation_probability=float(np.sum(p_red[:,None,None]*((lo+hi)>1e-9))/(24*len(crit))),
        vmin=float(v.min()),
        vmax=float(v.max()),
        peak_import=float((p_red[:,None]*gimp).sum(0).max())
    )

def evaluate_no_ess(kv=50000,export_frac=.15,beta=.95,vmin=.94,vmax=1.06):
    gimp=np.maximum(net_total,0)
    gexp=np.maximum(-net_total,0)
    v=np.sqrt(np.maximum(base_v,0))
    lo=np.maximum(0,vmin-v)
    hi=np.maximum(0,v-vmax)
    lo2=np.maximum(0,vmin**2-base_v)
    hi2=np.maximum(0,base_v-vmax**2)

    cost=np.array([
        np.sum(price*gimp[s]-export_frac*price*gexp[s])
        for s in range(20)
    ])
    risk=kv/len(crit)*np.sum(lo2+hi2,axis=(1,2))
    cc,_=weighted_cvar(cost,p_red,beta)
    cr,_=weighted_cvar(risk,p_red,beta)

    return dict(
        expected_cost=float(p_red@cost),
        expected_risk=float(p_red@risk),
        cvar_cost=cc,
        cvar_risk=cr,
        violation_pu_h=float(np.sum(p_red[:,None,None]*(lo+hi))),
        violation_probability=float(np.sum(p_red[:,None,None]*((lo+hi)>1e-9))/(24*len(crit))),
        vmin=float(v.min()),
        vmax=float(v.max()),
        peak_import=float((p_red[:,None]*gimp).sum(0).max()),
        v=v,
        gimp=gimp
    )

case1=evaluate_no_ess()
r2,i2,p2=solve_dispatch(lambda_risk=0.0,beta=.95)
case2=evaluate(r2,i2,p2)
r3,i3,p3=solve_dispatch(lambda_risk=0.9,beta=.95)
case3=evaluate(r3,i3,p3)

summary=pd.DataFrame([
    ['Case 1: no ESS',case1['expected_cost'],case1['cvar_cost'],case1['cvar_risk'],case1['violation_pu_h'],case1['vmin'],case1['peak_import']],
    ['Case 2: expected-value ESS',case2['expected_cost'],case2['cvar_cost'],case2['cvar_risk'],case2['violation_pu_h'],case2['vmin'],case2['peak_import']],
    ['Case 3: CVaR ESS',case3['expected_cost'],case3['cvar_cost'],case3['cvar_risk'],case3['violation_pu_h'],case3['vmin'],case3['peak_import']],
],columns=[
    'case',
    'expected economic cost',
    'CVaR economic cost',
    'CVaR network risk',
    'voltage violation p.u.-h',
    'minimum voltage',
    'peak expected import MW'
])
print(summary.round(4).to_string(index=False))

# ESS sequence and SOC.
fig,ax=plt.subplots(figsize=(10,4))
for label,case in [('Case 2 expected',case2),('Case 3 CVaR',case3)]:
    net=(case['pdis']-case['pch']).sum(axis=0)
    ax.step(hours,net,where='mid',label=label)
ax.axhline(0,linewidth=1)
ax.set_xlabel('Hour')
ax.set_ylabel('ESS net discharge power (MW)')
ax.grid(alpha=.25)
ax.legend()
plt.show()

fig,ax=plt.subplots(figsize=(10,4))
for k,name in enumerate(['ESS@15','ESS@32']):
    ax.plot(np.arange(25),case2['soc'][k],marker='o',alpha=.7,label=f'{name} - Case2')
    ax.plot(np.arange(25),case3['soc'][k],marker='x',alpha=.9,label=f'{name} - Case3')
ax.set_xlabel('Hour boundary')
ax.set_ylabel('Stored energy (MWh)')
ax.grid(alpha=.25)
ax.legend(ncol=2)
plt.show()

# Worst-scenario voltage comparison.
worst2=case2['v'].min(axis=(0,2))
worst3=case3['v'].min(axis=(0,2))
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(hours,worst2,marker='o',label='Case 2 worst-scenario min V')
ax.plot(hours,worst3,marker='o',label='Case 3 CVaR worst-scenario min V')
ax.axhline(.94,linestyle='--',label='0.94 p.u. reference')
ax.set_xlabel('Hour')
ax.set_ylabel('Minimum voltage (p.u.)')
ax.grid(alpha=.25)
ax.legend()
plt.show()

# Risk-cost sensitivity.
rows=[]
for lam in [0,.25,.5,.75,.9]:
    rr,ii,pp=solve_dispatch(lambda_risk=lam,beta=.95)
    ee=evaluate(rr,ii,pp)
    rows.append([
        lam,
        ee['expected_cost'],
        ee['cvar_risk'],
        ee['violation_pu_h'],
        ee['vmin']
    ])

sens_df=pd.DataFrame(rows,columns=[
    'risk weight λ',
    'expected cost',
    'CVaR network risk',
    'voltage violation p.u.-h',
    'minimum voltage'
])
print(sens_df.round(4).to_string(index=False))

fig,ax=plt.subplots(figsize=(6,4))
ax.plot(sens_df['expected cost'],sens_df['CVaR network risk'],marker='o')
for _,r in sens_df.iterrows():
    ax.annotate(
        f"λ={r['risk weight λ']}",
        (r['expected cost'],r['CVaR network risk'])
    )
ax.set_xlabel('Expected economic cost')
ax.set_ylabel('CVaR network risk')
ax.grid(alpha=.25)
plt.show()


## 3. 如何解读复现结果

运行全部单元后应重点看三类结果：

- **Case 1 → Case 2**：加入储能后，平均购电成本/峰值购电应下降；
- **Case 2 → Case 3**：加入 CVaR 后，平均经济成本可能略升，但尾部网络风险与最差电压应改善；
- **风险权重扫描**：随着风险权重增加，可直接看到“平均经济性 ↔ 尾部安全性”的 trade-off。

这与原论文 Table 4/5 的核心方向一致，但这里不声称逐小数点复现。原论文缺少历史 WT/PV/load 原始数据、TSL 参数、完整 modified-33-bus 联络开关与全部灵活资源参数、若干权重以及官方 MATLAB/YALMIP 实现。

### 原论文 Table 4（用于方向性核对）

| Case | fm | fr,c | fr,o | fr |
|---|---:|---:|---:|---:|
| Case 1 | 17969 | 20702 | 791 | 21493 |
| Case 2 | 11409 | 14402 | 223 | 14625 |
| Case 3 | 12768 | 13075 | 42 | 13117 |

核心现象：**Case 3 相比 Case 2，平均综合运行成本上升，但高风险场景损失下降，尤其电压相关尾部风险显著下降。**
